# Week 14 실습 · AI 디버깅 · 테스트 · 리팩토링

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week14/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. **재현 → 범위 좁히기 → 가설 → 확인 → 수정 → 회귀 테스트**의 디버깅 절차를 따릅니다.
2. 오류가 **나지 않는** 버그(조용한 버그)를 손으로 계산 가능한 **작은 데이터**로 찾아냅니다.
3. 기대 결과를 먼저 적는 **테스트**를 만들고, 테스트로 수정과 리팩토링의 안전을 확인합니다.
4. 이름 바꾸기·상수화·함수 추출·중복 제거·부수효과 제거로 코드를 **리팩토링**합니다.
5. AI에게 디버깅을 맡길 때의 **좋은 질문**과, AI 수정이 **새 버그를 만들지 않았는지** 검증하는 법을 익힙니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | 디버깅 절차와 트레이스백 | 15분 |
| 2 | Level 1 흔한 조용한 버그 | 15분 |
| 3 | Level 3 결함 있는 분석 코드 진단 | 30분 |
| 4 | Level 2~3 리팩토링 | 25분 |
| 5 | Level 4 AI 디버깅 | 20분 |
| 6 | 최종 프로젝트 점검 · 정리 | 15분 |

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request
import traceback

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

CAFE = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])


def run_tests(tests):
    """테스트 함수 목록을 실행해 통과/실패를 보여 준다."""
    passed = 0
    for t in tests:
        try:
            t()
            print(f"✅ {t.__name__}")
            passed += 1
        except AssertionError as e:
            print(f"❌ {t.__name__}: {e}")
        except Exception as e:
            print(f"💥 {t.__name__}: {type(e).__name__}: {e}")
    print(f"통과 {passed}/{len(tests)}")

---
## 1. 디버깅 절차와 트레이스백

| 단계 | 할 일 | 도구 |
|---|---|---|
| ① 재현 | 같은 입력으로 **항상** 같은 문제가 나오게 | 작은 입력, 시드 고정 |
| ② 범위 좁히기 | 어느 줄·어느 단계부터 틀렸나? | 중간값 `print`, 반으로 나누기 |
| ③ 가설 | "이 변수가 문자열이라서?" | 오류 메시지, 자료형 확인 |
| ④ 확인 | 가설이 맞는지 **실험** | `assert`, `type()`, `shape` |
| ⑤ 수정 | 원인을 고친다 (증상만 가리지 않는다) | |
| ⑥ 회귀 테스트 | 같은 버그가 다시 생기지 않도록 테스트 추가 | `test_` 함수 |

### 트레이스백은 아래에서 위로

In [ ]:
def load_prices(rows):
    return [parse_price(r) for r in rows]

def parse_price(text):
    return int(text.replace(",", ""))

def average_price(rows):
    prices = load_prices(rows)
    return sum(prices) / len(prices)

try:
    average_price(["2,500", "3,200", "3천원"])
except ValueError:
    traceback.print_exc(limit=4)

- **마지막 줄**: `ValueError: invalid literal for int() with base 10: '3천원'` → 무엇이 잘못됐나
- **그 바로 위**: `parse_price`의 `int(...)` → 어디서 터졌나
- **더 위로**: `load_prices` ← `average_price` → 누가 불렀나 (호출 경로)

> 라이브러리 내부 줄이 길게 나와도, **내가 쓴 코드의 마지막 줄**을 찾는 것이 출발점입니다.

---
## 2. Level 1 따라하기: 흔한 "조용한 버그"

오류 메시지가 **나지 않고 틀린 값만** 나오는 버그가 가장 위험합니다. 손으로 답을 알 수 있는 **작은 입력**이 최고의 무기입니다.

| 유형 | 예 | 잡는 법 |
|---|---|---|
| 경계 (off-by-one) | `range(1, n)`이 n을 빠뜨림 | 마지막 원소가 포함됐는지 |
| 분모 착오 | 영업일 대신 7로 나눔 | 손계산한 평균과 비교 |
| 집계 단위 | 건수로 셀 것을 수량으로 셈 | 작은 표로 직접 세기 |
| 부수효과 | 함수가 입력 DataFrame에 열을 추가 | 호출 전후 원본 비교 |
| 누적 초기화 위치 | 반복 밖에서 0으로 만들어야 할 것을 안에서 | 둘 이상의 그룹으로 실험 |
| NaN 비교 | `x == np.nan`은 항상 False | `isna()` |

In [ ]:
# 누적 초기화 위치 버그: 학과별 합계를 구하려 했다
pairs = [("A", 10), ("A", 20), ("B", 5)]
totals = {}
for dept in ["A", "B"]:
    for d, v in pairs:
        total = 0                     # ← 버그: 반복 안에서 매번 0
        if d == dept:
            total += v
    totals[dept] = total
print(totals, "← 기대: {'A': 30, 'B': 5}")

In [ ]:
# 범위 좁히기: 중간값을 찍어 보면 바로 보인다
for dept in ["A"]:
    for d, v in pairs:
        total = 0
        if d == dept:
            total += v
        print(f"  d={d}, v={v}, total={total}")

---
## 3. Level 3 적용하기: 결함 있는 분석 코드 진단

AI가 만들어 준 **주간 보고서** 함수입니다. 요구사항은 다음과 같았습니다.

> 주(ISO 주차)마다 **영업일 기준 일평균 매출**과 **판매 수량이 가장 많은 메뉴**를 구한다. 입력 DataFrame은 바꾸지 않는다. 데이터의 첫 주부터 마지막 주까지 모두 포함한다.

오류 없이 실행되지만, 정말 맞을까요?

In [ ]:
def weekly_report_ai(cafe):
    cafe["revenue"] = cafe["unit_price"] * cafe["qty"]
    cafe["week"] = cafe["date"].dt.isocalendar().week
    result = {}
    for w in range(cafe["week"].min(), cafe["week"].max()):
        sub = cafe[cafe["week"] == w]
        avg = sub["revenue"].sum() / 7
        top = sub["menu"].value_counts().index[0]
        result[w] = (round(avg), top)
    return result

report = weekly_report_ai(CAFE.copy())
print(len(report), "주", list(report.items())[:2])

### ① 재현 + 작은 데이터: 손으로 답을 알 수 있는 입력

2주, 3일치, 메뉴 2개로 **기대 결과를 먼저** 계산해 둡니다.

| 날짜 | 주차 | 메뉴 | 가격 | 수량 |
|---|---|---|---|---|
| 2025-03-03 (월) | 10 | A | 1000 | 1 |
| 2025-03-03 (월) | 10 | B | 2000 | 1 |
| 2025-03-04 (화) | 10 | A | 1000 | 1 |
| 2025-03-10 (월) | 11 | B | 2000 | 3 |
| 2025-03-10 (월) | 11 | A | 1000 | 1 |
| 2025-03-10 (월) | 11 | A | 1000 | 1 |

- 10주: 매출 4,000원 ÷ 영업일 **2일** = 2,000원, 수량 최다 A(2개)
- 11주: 매출 8,000원 ÷ 영업일 **1일** = 8,000원, 수량 최다 **B(3개)** (건수로는 A가 2건)

In [ ]:
def make_tiny():
    return pd.DataFrame({
        "date": pd.to_datetime(["2025-03-03", "2025-03-03", "2025-03-04", "2025-03-10", "2025-03-10", "2025-03-10"]),
        "menu": ["A", "B", "A", "B", "A", "A"],
        "unit_price": [1000, 2000, 1000, 2000, 1000, 1000],
        "qty": [1, 1, 1, 3, 1, 1],
    })

EXPECTED = {10: (2000, "A"), 11: (8000, "B")}


def make_tests(fn):
    def test_values():
        got = fn(make_tiny())
        assert got == EXPECTED, f"기대 {EXPECTED}, 결과 {got}"

    def test_last_week_included():
        assert 11 in fn(make_tiny()), "마지막 주(11주)가 빠졌다"

    def test_input_unchanged():
        tiny = make_tiny()
        before = tiny.copy()
        fn(tiny)
        assert tiny.equals(before), f"입력이 바뀌었다: 새 열 {set(tiny.columns) - set(before.columns)}"

    return [test_values, test_last_week_included, test_input_unchanged]


run_tests(make_tests(weekly_report_ai))

### ② 범위 좁히기와 ③ 가설

In [ ]:
tiny = make_tiny()
out = weekly_report_ai(tiny)
print("결과:", out)
print("주차 범위:", list(range(tiny["week"].min(), tiny["week"].max())), "← 11이 없다 (range 끝은 미만)")
print("10주 매출 합:", tiny.loc[tiny["week"] == 10, "revenue"].sum(), "/ 7 =", 4000 / 7, "← 영업일(2)로 나눠야")
print("11주 건수 기준:", tiny.loc[tiny["week"] == 11, "menu"].value_counts().to_dict(), "← 수량 기준이어야")

| 버그 | 증상 | 원인 | 수정 |
|---|---|---|---|
| 1 | 마지막 주 누락 | `range(min, max)`는 max 미만 | 실제로 있는 주차를 `unique()`로 순회 |
| 2 | 일평균이 너무 작음 | 영업일이 아니라 7로 나눔 | `sub["date"].nunique()` |
| 3 | 인기 메뉴가 다름 | 건수(`value_counts`) 기준 | `groupby("menu")["qty"].sum().idxmax()` |
| 4 | 입력이 바뀜 | `cafe["revenue"] = ...` 원본 수정 | `cafe.copy()` 또는 `assign` |

### ④ 수정 → ⑤ 회귀 테스트

In [ ]:
def weekly_report_fixed(cafe):
    df = cafe.assign(revenue=cafe["unit_price"] * cafe["qty"],
                     week=cafe["date"].dt.isocalendar().week)
    result = {}
    for w in sorted(df["week"].unique()):
        sub = df[df["week"] == w]
        avg = sub["revenue"].sum() / sub["date"].nunique()
        top = sub.groupby("menu")["qty"].sum().idxmax()
        result[int(w)] = (round(avg), top)
    return result

run_tests(make_tests(weekly_report_fixed))

In [ ]:
full = weekly_report_fixed(CAFE)
print(len(full), "주 (AI 버전은", len(report), "주)")
print("첫 주:", list(full.items())[0], "/ 마지막 주:", list(full.items())[-1])
assert len(CAFE.columns) == 9          # 원본 그대로

> **회귀 테스트**: 고친 뒤에도 `make_tests`를 계속 남겨 둡니다. 나중에 코드를 바꾸다 같은 버그가 다시 생기면 테스트가 바로 알려 줍니다.

---
## 4. 리팩토링: 동작은 그대로, 읽기 쉽게

리팩토링은 **겉으로 보이는 결과를 바꾸지 않고** 코드 구조를 개선하는 일입니다. 그래서 **테스트가 먼저** 있어야 안전합니다.

| 기법 | 전 | 후 |
|---|---|---|
| 의미 있는 이름 | `d`, `x2`, `tmp` | `daily`, `ice_ratio` |
| 매직 넘버 → 상수 | `* 0.3`, `< 12` | `WEIGHTS`, `MIN_ATTENDANCE` |
| 함수 추출 | 긴 함수 하나 | 작은 함수 여러 개 |
| 중복 제거 | 같은 코드 3번 | 반복문 또는 함수 하나 |
| 부수효과 제거 | 입력 수정 | 새 객체 반환 |
| early return | 깊은 if 중첩 | 예외 상황 먼저 return |

### 리팩토링 전: 동작은 하지만 읽기 어려운 코드

In [ ]:
def f(df):
    r = {}
    x = df[df["category"] == "커피"]
    r["커피"] = (len(x), int((x["unit_price"] * x["qty"]).sum()), round((x["unit_price"] * x["qty"]).sum() / len(x)))
    x = df[df["category"] == "논커피"]
    r["논커피"] = (len(x), int((x["unit_price"] * x["qty"]).sum()), round((x["unit_price"] * x["qty"]).sum() / len(x)))
    x = df[df["category"] == "디저트"]
    r["디저트"] = (len(x), int((x["unit_price"] * x["qty"]).sum()), round((x["unit_price"] * x["qty"]).sum() / len(x)))
    return r

before = f(CAFE)
before

### 리팩토링 후

In [ ]:
CATEGORIES = ["커피", "논커피", "디저트"]


def summarize_group(rows):
    """주문 행들의 (주문 건수, 매출 합계, 건당 평균 매출)."""
    revenue = rows["unit_price"] * rows["qty"]
    return len(rows), int(revenue.sum()), round(revenue.sum() / len(rows))


def category_summary(cafe, categories=CATEGORIES):
    """분류별 (주문 건수, 매출 합계, 건당 평균 매출) 딕셔너리."""
    return {cat: summarize_group(cafe[cafe["category"] == cat]) for cat in categories}


after = category_summary(CAFE)
assert after == before, "리팩토링 후 결과가 달라졌다!"
print("결과 동일 ✅", after)

### ✏️ 직접 해보기 (Level 2)

1. `summarize_group`에 빈 DataFrame을 넣으면 어떻게 되나요? (NumPy 숫자 `0 / 0`은 `ZeroDivisionError`가 아니라 경고와 함께 `nan`이 되고, `round(nan)`에서 `ValueError`가 납니다.) 오류 대신 `(0, 0, 0)`을 돌려주도록 고치고, 그 경우를 확인하는 테스트를 추가해 보세요.
2. `category_summary`를 pandas의 `groupby().agg()`로 다시 써 보고, `assert`로 결과가 같은지 확인하세요.

In [ ]:
import warnings

empty = CAFE.iloc[0:0]
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")          # 0/0 경고 숨기기
        print(summarize_group(empty))
except (ZeroDivisionError, ValueError) as e:
    print(f"{type(e).__name__}: {e}  ← 고쳐 보세요")

---
## 5. Level 4 AI와 함께: 디버깅을 맡길 때

### 나쁜 질문 vs 좋은 질문

| ❌ | ✅ |
|---|---|
| "코드가 이상해요. 고쳐 주세요." (코드 300줄 붙여 넣기) | **최소 재현 코드** (10~20줄) |
| | **입력**과 **기대 결과**, **실제 결과** |
| | 오류라면 **트레이스백 전체** |
| | 이미 확인한 것 ("자료형은 int임을 확인함") |
| | 요구: "원인을 먼저 설명하고, 수정은 최소한으로" |

```text
아래 함수는 주별 일평균 매출을 구해야 합니다.
[최소 재현 코드 + make_tiny() 데이터]
기대 결과: {10: (2000, 'A'), 11: (8000, 'B')}
실제 결과: {10: (571, 'A')}
11주가 빠지고 10주 평균이 작습니다. 원인을 먼저 줄 단위로 설명하고,
수정은 최소한으로 해 주세요. 입력 DataFrame은 바꾸면 안 됩니다.
```

### ⚠️ AI 수정을 검증하라: 테스트를 "통과시키는" 가짜 수정

AI에게 "테스트가 통과하게 고쳐 줘"라고만 하면, 가끔 이런 답이 옵니다.

In [ ]:
def weekly_report_cheat(cafe):
    if len(cafe) == 6:                         # 테스트 데이터일 때만…
        return {10: (2000, "A"), 11: (8000, "B")}
    return weekly_report_ai(cafe.copy())

run_tests(make_tests(weekly_report_cheat))

테스트는 모두 통과하지만 **실제 데이터에는 여전히 틀린 코드**입니다. 그래서

1. AI가 고친 **코드 자체를 읽고** 한 줄씩 설명할 수 있어야 하고,
2. 테스트 데이터를 **하나 더**(다른 크기·다른 값) 만들어 확인하며,
3. AI가 **테스트 코드를 수정**하자고 하면 특히 의심합니다.

In [ ]:
def make_tiny2():
    t = make_tiny()
    extra = pd.DataFrame({"date": pd.to_datetime(["2025-03-11"]), "menu": ["A"], "unit_price": [1000], "qty": [5]})
    return pd.concat([t, extra], ignore_index=True)       # 11주: 매출 13,000 / 2일, 수량 최다 A(7개)


for fn in [weekly_report_cheat, weekly_report_fixed]:
    got = fn(make_tiny2())
    print(f"{fn.__name__:<22}", got, "✅" if got == {10: (2000, "A"), 11: (6500, "A")} else "❌")

---
## 6. 최종 프로젝트 점검 · 정리

### 릴리스 후보(14주) 체크리스트

- [ ] **런타임 → 다시 시작 및 모두 실행**에서 오류 없음
- [ ] 핵심 계산 함수마다 **작은 데이터 테스트**가 1개 이상 있음
- [ ] 데이터 정제·집계 함수는 **입력을 바꾸지 않음**
- [ ] 매직 넘버는 상수로, 함수 이름은 하는 일을 설명
- [ ] 같은 코드가 3번 이상 반복되지 않음
- [ ] AI가 만든 코드는 모두 **한 줄씩 설명할 수 있음**, AI 활용 기록 갱신

### 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 디버깅 절차 | 재현 → 범위 좁히기 → 가설 → 확인 → 수정 → 회귀 테스트 |
| 트레이스백 | 아래에서 위로, 내 코드의 마지막 줄 |
| 조용한 버그 | 손계산 가능한 작은 데이터로 기대 결과 먼저 |
| 테스트 | 값 · 경계 · 입력 불변, 테스트 팩토리 `make_tests(fn)` |
| 리팩토링 | 이름 · 상수 · 함수 추출 · 중복 제거, **결과 동일 assert** |
| AI 디버깅 | 최소 재현 + 기대/실제 + 트레이스백, 가짜 수정 경계, 두 번째 테스트 데이터 |

### 다음 주 예고

**Week 15 · 최종 프로젝트 발표와 재현성**: 노트북을 처음부터 끝까지 재현 가능하게 다듬고 5분 발표를 합니다.